# Test KL minimization with non-invertible transformation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from gpsr.beams import NNDist

In [ ]:
ndim = 2
nsamp = 10_000

model_dist = NNDist(width=32, depth=2, ndim=ndim)
targ_dist = torch.distributions.MultivariateNormal(
    torch.zeros(ndim),
    torch.eye(ndim),
)
optimizer = torch.optim.Adam(model_dist.parameters(), lr=0.005)

# Warmup
for i in range(300):
    x = model_dist.sample(nsamp)
    cov_matrix = torch.cov(x.T)
    loss = torch.mean(torch.abs(cov_matrix - torch.eye(ndim)))
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

history = {"loss": []}

for iteration in range(201):
    loss = -model_dist.entropy(nsamp, prior=targ_dist)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    history["loss"].append(loss.item())

    if iteration % 50 == 0:
        print(iteration, loss)

In [ ]:
fig, ax = plt.subplots(figsize=(3.0, 2.0))
ax.plot(history["loss"])
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
plt.show()

In [ ]:
with torch.no_grad():
    # Sample and evaluate density at each point
    x, log_p = model_dist.sample_and_log_prob(100_000)

    x = x.numpy()
    log_p = log_p.numpy()

    x_targ = targ_dist.rsample((x.shape[0],))
    x_targ = x_targ.numpy()

    # Sort by density
    idx = np.argsort(log_p)
    x = x[idx]
    log_p = log_p[idx]

    # Plot histogram and scatter plot with points colored by density
    plot_xmax = 4.0
    plot_limits = 2 * [(-plot_xmax, plot_xmax)]

    fig, axs = plt.subplots(ncols=3, sharex=True, sharey=True, figsize=(8.5, 2.5))

    grid_values, grid_edges = np.histogramdd(
        x, bins=64, range=plot_limits, density=True
    )
    axs[0].hist2d(x[:, 0], x[:, 1], bins=64, range=plot_limits, density=True)
    axs[1].scatter(x[:, 0], x[:, 1], c=np.exp(log_p), s=1)
    axs[2].hist2d(x_targ[:, 0], x_targ[:, 1], bins=64, range=plot_limits, density=True)
    axs[0].set_title("Model samples", fontsize="medium")
    axs[1].set_title("Model density", fontsize="medium")
    axs[2].set_title("Target samples", fontsize="medium")
    plt.show()

### 6D

In [ ]:
ndim = 6
nsamp = 10_000

model_dist = NNDist(width=32, depth=2, ndim=ndim)
targ_dist = torch.distributions.MultivariateNormal(
    torch.zeros(ndim),
    torch.eye(ndim),
)
optimizer = torch.optim.Adam(model_dist.parameters(), lr=0.001)

# Warmup
for i in range(500):
    x = model_dist.sample(nsamp)
    cov_matrix = torch.cov(x.T)
    loss = torch.mean(torch.abs(cov_matrix - torch.eye(ndim)))
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

history = {"loss": []}

for iteration in range(501):
    loss = -model_dist.entropy(nsamp, prior=targ_dist)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    history["loss"].append(loss.item())

    if iteration % 50 == 0:
        print(iteration, loss)

In [ ]:
fig, ax = plt.subplots(figsize=(3.0, 2.0))
ax.plot(history["loss"])
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
plt.show()

This time we won't make the scatter plot: the density at each point is in 6D space, not the 2D space in the plot.

In [ ]:
with torch.no_grad():
    # Sample particles.
    x_model = model_dist.sample(100_000)
    x_targ = targ_dist.rsample((x_model.shape[0],))

    # Plot histogram and scatter plot with points colored by density.
    plot_xmax = 4.0
    plot_limits = 2 * [(-plot_xmax, plot_xmax)]

    fig, axs = plt.subplots(ncols=2, sharex=True, sharey=True, figsize=(5.5, 2.5))
    for ax, x in zip(axs, [x_model, x_targ]):
        ax.hist2d(x[:, 0], x[:, 1], bins=64, range=plot_limits, density=True)
    axs[0].set_title("Model", fontsize="medium")
    axs[1].set_title("Target", fontsize="medium")
    plt.show()